In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIXED #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter



###########################
# Data Manipulation / Model
###########################
# FIXED update with your username and password and CRUD Python module name. NOTE: You will
# likely need more variables for your constructor to handle the hostname and port of the MongoDB
# server, and the database and collection names

username = "aacuser"
password = "SNHU1234"
shelter = AnimalShelter(username, password)


# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)


## Debug
#print(len(df.to_dict(orient='records')))
#print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Alex Abercrombie CS-340 Dashboard'))),
    html.Hr(),
    dcc.Dropdown(
        id = "filter-dropdown",
        options = [
            {"label" : "Show All", "value" : "all"} ] +
        [
            {"label": i, "value": i} for i in df["animal_type"].dropna().unique()
        ],
        placeholder = "Filter by animal type"
    ),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        row_selectable = "single",
        selected_rows = [0],
        #FIXED: Set up the features for your interactive data table to make it user-friendly for your client
        sort_action ="native",      
        filter_action = "native",
        page_size = 50
    ),
    html.Br(),
    html.Hr(),
    html.Div(
            id='map-id',
            className='col s12 m6',
            )
])

#############################################
# Interaction Between Components / Controller
#############################################
#This callback will highlight a row on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)

def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-dropdown', 'value')]
)

def filter_table(selected_value):
    if selected_value is None or selected_value == "all":
        return df.to_dict("records")
    filtered_df = df[df["animal_type"] == selected_value]
    return filtered_df.to_dict("records")


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    
    if viewData is None:
        return []
    #FIXED Add in the code for your geolocation chart
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can 
    # be converted to a row index here
    
    if index is None:
        row = 0
    else: 
        row = index[0]
        

    if pd.isna(dff.iloc[row,9]) or dff.iloc[row,9] == "":
        animal_name = "No Name"
    else:
        animal_name = dff.iloc[row,9]

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75,-97.48], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
           # Marker with tool tip and popup
           # Column 13 and 14 define the grid-coordinates for 
           # the map
           # Column 4 defines the breed for the animal
           # Column 9 defines the name of the animal
                   dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
                             children=[
                                 dl.Tooltip(animal_name),
                                 dl.Popup([
                                     html.H1(animal_name),
                                     html.P([html.B("Type: "), dff.iloc[row, 3]]),
                                     html.P([html.B("Breed: "), dff.iloc[row,4]]),
                                     html.P([html.B("Color: "), dff.iloc[row,5]]),
                                     html.P([html.B("Outcome: "), dff.iloc[row,12]])
                                 ])
                             ])
               ])
    ]

    
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode="inline")